In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Data;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using System.Diagnostics;
using Microsoft.AspNetCore.Html;
using System.Text.RegularExpressions;
using BoSSS.Application.SipPoisson;
Init();

In [ ]:
int[] degS = {2, 3, 4, 5};
int[] setupS = {0, 1, 2, 3, 4, 5, 6};
double[,] slopes = new double[setupS.Length,degS.Length];
BoSSS.Application.SipPoisson.Tests.TestProgram.FailAssertion = false;

In [ ]:
for(int i = 0; i < setupS.Length; i++){
    for(int j = 0; j < degS.Length; j++){
        slopes[i,j] = BoSSS.Application.SipPoisson.Tests.TestProgram.TestOperatorConvergenceCustom(degS[j], setupS[i])[0];
    }
}

In [ ]:
static void WriteTexTable(double[,] slopes, int[] setupS,  int[] degS){

    string WriteEOC(int i){
        return " & " + slopes[i,0].ToString("N2") + " & " + slopes[i,1].ToString("N2") + " & " + slopes[i,2].ToString("N2") +" & " + slopes[i,3].ToString("N2");
    }
    
    using(var stw = new StringWriter()) {    
        stw.WriteLine(@"\begin{table}[t!]");
        stw.WriteLine(@"    \centering");
        stw.WriteLine(@"    \caption{EOC, for the steady heat equation}");
        stw.WriteLine(@"    \label{tab:singlephasebc:1}");
        stw.WriteLine(@"    \begin{tabularx}{\linewidth}{ >{\hsize=2\hsize \raggedleft\arraybackslash}X >{\hsize=2\hsize\raggedleft\arraybackslash}X | >{\hsize=0.5\hsize\raggedleft\arraybackslash}X >{\hsize=0.5\hsize\raggedleft\arraybackslash}X >{\hsize=0.5\hsize\raggedleft\arraybackslash}X >{\hsize=0.5\hsize\raggedleft\arraybackslash}X }		");
        stw.WriteLine(@"        $\boundary_u$ & $\boundary_l$ & $\Pdeg="+degS[0]+@"$ & $\Pdeg="+degS[1]+@"$ & $\Pdeg="+degS[2]+@"$ & $\Pdeg="+degS[3]+@"$ \\");
        stw.WriteLine(@"        $\temp=\sin(x)e^y$ & $\temp=\sin(x)e^y$"+WriteEOC(0)+@"\\");
        stw.WriteLine(@"        $\temp=\ln(r), r=\sqrt{x^2+y^2}$ & $\temp=\ln(r)$ "+WriteEOC(1)+@"\\");
        stw.WriteLine(@"        $\temp=\ln(r), r=\sqrt{x^2+(y+0.01)^2}$ & $\temp=\ln(r)$"+WriteEOC(2)+@"\\");
        stw.WriteLine(@"        $\temp=\ln(r), r=\sqrt{x^2+(y+0.1)^2}$ & $\temp=\ln(r)$"+WriteEOC(3)+@"\\");
        stw.WriteLine(@"        $\temp=\ln(r), r=\sqrt{x^2+(y+1)^2}$ & $\temp=\ln(r)$"+WriteEOC(4)+@"\\");
        stw.WriteLine(@"        $\temp=0$ & $\temp=\sin(2\pi(x-R)/(2R))$"+WriteEOC(5)+@"\\");
        stw.WriteLine(@"        $\temp=0$ & \cref{eq:bndycondT}"+WriteEOC(6)+@"\\");
        stw.WriteLine(@"    \end{tabularx}");
        stw.WriteLine(@"\end{table}	");
        
        File.WriteAllText($"singlephaseboundarycond.txt", stw.ToString());
    }
}

Export table

In [ ]:
//WriteTexTable(slopes, setupS, degS);

Assert slopes to state of dissertation $\pm 0.1$

In [ ]:
double[][] slopesExpected = new double[setupS.Length][];
slopesExpected[0] = new double[] {3.27, 4.15, 5.04, 6.15};
slopesExpected[1] = new double[] {0.84, 0.91, 0.9, 0.91};
slopesExpected[2] = new double[] {1.34, 1.62, 1.81, 1.89};
slopesExpected[3] = new double[] {2.26, 2.86, 3.46, 4.13};
slopesExpected[4] = new double[] {3.24, 4.17, 4.88, 5.89};
slopesExpected[5] = new double[] {2.94, 3.35, 3.30, 3.19};
slopesExpected[6] = new double[] {3.02, 3.9, 4.7, 4.88};


In [ ]:
bool success = true;
double thrsh = 0.1;

for(int i = 0; i < setupS.Length; i++){

    for(int j = 0; j < degS.Length; j++){
        if(slopes[i,j] < slopesExpected[i][j] - thrsh || slopes[i,j] > slopesExpected[i][j] + thrsh){
            success &= false;
            Console.WriteLine("Mismatch in computed slopes for setup {0}, for degree {1}", setupS[i], degS[j]);
        }
    }
}

if(!success){
    throw new ApplicationException("Not all slopes match!");
}